# TCS Equity Research — Context Management Deep Dive
## NucleusIQ v0.7.6 | Real Data, A/B Comparison, Stress Test, Two Modes, Memory

---

### What This Notebook Proves

A **real NucleusIQ Agent** reads **actual TCS annual report PDFs** (~340 pages each),
fetches **live financial data from the web**, and generates a comprehensive equity research report.

**The key question: What BENEFIT does context management provide?**

We answer this with **three runs** of the same task:

| Run | Context Management | Purpose |
|---|---|---|
| **A. Managed (200K budget)** | Full: offloading, masking, compaction | Normal production config |
| **B. Baseline (no context mgmt)** | Completely OFF | Control group — see raw behavior |
| **C. Stress Test (30K budget)** | Aggressive: tight budget forces compaction | Simulates small-model constraints |

**Nine things demonstrated:**

1. **A/B comparison** — Same task WITH and WITHOUT context management, concrete token/cost numbers
2. **Stress test** — 30K budget forces compaction pipeline to fire, proving it works under pressure
3. **Real tools** — PDF reader (pypdf), web fetcher (requests), not hardcoded data
4. **Standard vs Autonomous modes** — same task, compared with full sub-agent telemetry rollup
5. **Context management** — ObservationMasker, offloading, compaction pipeline in action
6. **Memory** — SlidingWindowMemory enables follow-up questions
7. **Explicit follow-up tracking** — Each follow-up shows TOOLS vs MEMORY ONLY source
8. **Turn-by-turn token growth** — Visual trace showing per-call token accumulation
9. **Clear dashboards** — Progress bars, region breakdown, cost impact

**Prerequisites:** TCS annual report PDFs in `notebooks/agents/data/tcs/`

---

## 1. Setup & Install

In [1]:
import subprocess, sys, pathlib

_root = pathlib.Path.cwd().resolve()
while not (_root / "src").exists() and _root != _root.parent:
    _root = _root.parent

subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(_root / "src" / "nucleusiq"), "--upgrade", "-q"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(_root / "src" / "providers" / "llms" / "openai"), "--upgrade", "-q"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv", "pypdf", "requests", "-q"])

for mod_name in list(sys.modules):
    if mod_name.startswith(("nucleusiq", "nucleusiq_openai")):
        del sys.modules[mod_name]

import nucleusiq, nucleusiq_openai
print(f"nucleusiq       : {nucleusiq.__file__}")
print(f"nucleusiq_openai: {nucleusiq_openai.__file__}")

nucleusiq       : C:\Users\Brijesh Singh\Documents\github-project\NucleusIQ\src\nucleusiq\core\__init__.py
nucleusiq_openai: C:\Users\Brijesh Singh\Documents\github-project\NucleusIQ\src\providers\llms\openai\nucleusiq_openai\__init__.py


In [2]:
import os, re, json, logging
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join("..", "..", ".env"))
except ImportError:
    pass

try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None

try:
    import requests as _requests
except ImportError:
    _requests = None

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env or environment"
print(f"API key loaded (first 8 chars): {os.getenv('OPENAI_API_KEY')[:8]}...")
print(f"pypdf available: {PdfReader is not None}")
print(f"requests available: {_requests is not None}")

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")
ctx_logger = logging.getLogger("nucleusiq.agents.context.engine")
ctx_logger.setLevel(logging.DEBUG)

API key loaded (first 8 chars): sk-proj-...
pypdf available: True
requests available: True


In [3]:
from nucleusiq.agents import Agent
from nucleusiq.agents.config import AgentConfig, ExecutionMode
from nucleusiq.agents.context.config import ContextConfig, ContextStrategy
from nucleusiq.agents.task import Task
from nucleusiq.tools.decorators import tool
from nucleusiq.memory.factory import MemoryFactory, MemoryStrategy
from nucleusiq.plugins.builtin.model_call_limit import ModelCallLimitPlugin
from nucleusiq.plugins.builtin.tool_call_limit import ToolCallLimitPlugin
from nucleusiq.plugins.builtin.tool_retry import ToolRetryPlugin
from nucleusiq.prompts.zero_shot import ZeroShotPrompt
from nucleusiq_openai import BaseOpenAI

MODEL = os.getenv("NUCLEUSIQ_MODEL", "gpt-4.1-mini")
llm = BaseOpenAI(model_name=MODEL, api_key=os.getenv("OPENAI_API_KEY"), timeout=300.0)
CTX_WINDOW = llm.get_context_window()
print(f"LLM: {llm.model_name}")
print(f"Context window: {CTX_WINDOW:,} tokens")
print(f"Max output: 32,768 tokens (~25K words)")
print(f"Timeout: 300s (5 min — needed for long synthesis generation)")
print(f"NOTE: GPT-4.1-mini — 1M context, 32K output, beats GPT-4o on many benchmarks.")
print(f"      Set NUCLEUSIQ_MODEL=gpt-4.1 for flagship (requires Tier 2+ for >30K TPM).")

LLM: gpt-4.1-mini
Context window: 1,047,576 tokens
Max output: 32,768 tokens (~25K words)
Timeout: 300s (5 min — needed for long synthesis generation)
NOTE: GPT-4.1-mini — 1M context, 32K output, beats GPT-4o on many benchmarks.
      Set NUCLEUSIQ_MODEL=gpt-4.1 for flagship (requires Tier 2+ for >30K TPM).


In [4]:
HERE = Path.cwd().resolve()
NUCLEUSIQ_ROOT = _root
AGENTS_DIR = NUCLEUSIQ_ROOT / "notebooks" / "agents"
DATA_TCS = AGENTS_DIR / "data" / "tcs"
DATA_TCS.mkdir(parents=True, exist_ok=True)
GITHUB_PROJECT = NUCLEUSIQ_ROOT.parent


def discover_tcs_pdfs() -> list[Path]:
    found: list[Path] = []
    seen: set[str] = set()

    def consider(path: Path) -> None:
        if not path.is_file():
            return
        key = str(path.resolve())
        if key in seen:
            return
        name = path.name.lower()
        if not name.endswith(".pdf"):
            return
        if "tcs" in name or "annual" in name or "integrated" in name:
            seen.add(key)
            found.append(path.resolve())

    if DATA_TCS.is_dir():
        for p in DATA_TCS.glob("*.pdf"):
            consider(p)
    for root_dir in (NUCLEUSIQ_ROOT, GITHUB_PROJECT):
        if not root_dir.is_dir():
            continue
        try:
            for p in root_dir.rglob("*.pdf"):
                if len(found) >= 6:
                    break
                consider(p)
        except OSError:
            pass
    return sorted(found, key=lambda x: x.name)


PDF_PATHS = discover_tcs_pdfs()
print(f"PDFs found: {len(PDF_PATHS)}")
for p in PDF_PATHS:
    print(f"  - {p.name}")
assert PDF_PATHS, f"No TCS PDFs found. Place them under {DATA_TCS}"

PDFs found: 2
  - tcs_annual_report-2024.pdf
  - tcs_annual_report-2025.pdf


---

## 2. Real Research Tools

These tools read **actual TCS annual report PDFs** from disk and fetch **live financial data**.
Nothing is hardcoded.

| Tool | What It Does | Data Source |
|------|-------------|------------|
| `list_tcs_pdf_inventory` | Discovers available PDF files | Local filesystem |
| `read_annual_report_excerpt` | Extracts text pages from PDFs | pypdf + real TCS reports |
| `fetch_allowlisted_finance_page` | Fetches financial data from allowed websites | screener.in, moneycontrol.com |

In [5]:
PDF_BY_BASENAME = {p.name: p for p in PDF_PATHS}
MAX_FETCH_BYTES = 200_000
MAX_EXCERPT_CHARS = 10_000

ALLOWED_DOMAIN_SUFFIXES = (
    "tcs.com", "moneycontrol.com", "nseindia.com",
    "economictimes.indiatimes.com", "screener.in",
)


def _host_allowed(netloc: str) -> bool:
    h = netloc.lower().removeprefix("www.")
    return any(h == s or h.endswith("." + s) for s in ALLOWED_DOMAIN_SUFFIXES)


def _strip_html(html: str) -> str:
    t = re.sub(r"<script[^>]*>.*?</script>", " ", html, flags=re.I | re.S)
    t = re.sub(r"<style[^>]*>.*?</style>", " ", t, flags=re.I | re.S)
    t = re.sub(r"<[^>]+>", " ", t)
    return re.sub(r"\s+", " ", t).strip()


@tool
def list_tcs_pdf_inventory() -> str:
    """List local TCS annual report PDFs available for read_annual_report_excerpt."""
    print("\n  >>> [TOOL] list_tcs_pdf_inventory()")
    if not PDF_PATHS:
        return "No PDFs discovered."
    lines = [f"- {p.name} ({p.stat().st_size // 1024}KB)" for p in PDF_PATHS]
    result = "Available PDFs:\n" + "\n".join(lines)
    print(f"  >>> [RESULT] {len(result)} chars — {len(PDF_PATHS)} PDFs found")
    return result


@tool
def read_annual_report_excerpt(
    filename: str,
    start_page: int = 1,
    max_pages: int = 10,
) -> str:
    """Read text from a local annual report PDF. Use list_tcs_pdf_inventory first.

    Args:
        filename: Exact PDF file name (basename), e.g. 'tcs_annual_report-2025.pdf'.
        start_page: 1-based start page.
        max_pages: Number of pages to read from start_page.
    """
    print(f"\n  >>> [TOOL] read_annual_report_excerpt('{filename}', start={start_page}, pages={max_pages})")
    if PdfReader is None:
        return "Error: install pypdf."
    base = Path(filename).name
    path = PDF_BY_BASENAME.get(base)
    if path is None:
        return f"Error: unknown file '{base}'. Call list_tcs_pdf_inventory for valid names."
    try:
        reader = PdfReader(str(path))
        n = len(reader.pages)
        start_idx = max(0, start_page - 1)
        end_idx = min(n, start_idx + max(1, max_pages))
        chunks: list[str] = []
        for i in range(start_idx, end_idx):
            t = reader.pages[i].extract_text() or ""
            chunks.append(f"--- Page {i + 1} ---\n{t}")
        text = "\n\n".join(chunks)
        if len(text) > MAX_EXCERPT_CHARS:
            text = text[:MAX_EXCERPT_CHARS] + "\n...[truncated]"
        print(f"  >>> [RESULT] {len(text):,} chars from pages {start_page}-{start_page + (end_idx - start_idx) - 1} of {n} total")
        return text or "(no text extracted)"
    except Exception as e:
        return f"Error reading PDF: {e}"


@tool
def fetch_allowlisted_finance_page(url: str) -> str:
    """Fetch HTML from an allowlisted finance/news URL and return plain text.

    Allowed domain suffixes: tcs.com, moneycontrol.com, nseindia.com,
    economictimes.indiatimes.com, screener.in
    """
    print(f"\n  >>> [TOOL] fetch_allowlisted_finance_page('{url[:80]}')")
    if _requests is None:
        return "Error: install requests."
    try:
        parsed = urlparse(url)
        if parsed.scheme not in ("http", "https") or not parsed.netloc:
            return "Error: invalid URL."
        if not _host_allowed(parsed.netloc):
            return "Error: host not allowlisted. Use only: " + ", ".join(ALLOWED_DOMAIN_SUFFIXES)
        headers = {"User-Agent": "NucleusIQ-ResearchShowcase/1.0 (educational demo)"}
        r = _requests.get(url, timeout=20, headers=headers, allow_redirects=True)
        r.raise_for_status()
        raw = r.content[:MAX_FETCH_BYTES]
        text = raw.decode("utf-8", errors="ignore")
        plain = _strip_html(text)
        if len(plain) > 12000:
            plain = plain[:12000] + " ...[truncated]"
        print(f"  >>> [RESULT] {len(plain):,} chars from {r.url}")
        return f"URL final: {r.url}\n\n{plain}"
    except Exception as e:
        return f"Error fetching URL: {e}"


ALL_TOOLS = [list_tcs_pdf_inventory, read_annual_report_excerpt, fetch_allowlisted_finance_page]
print(f"Tools ready: {[getattr(t, 'name', type(t).__name__) for t in ALL_TOOLS]}")

Tools ready: ['list_tcs_pdf_inventory', 'read_annual_report_excerpt', 'fetch_allowlisted_finance_page']


---

## 3. Shared Configuration

Both Standard and Autonomous agents share the same context config and task.

In [6]:
CTX_CONFIG = ContextConfig(
    optimal_budget=200_000,
    response_reserve=32_768,
    tool_result_threshold=500,
    strategy=ContextStrategy.PROGRESSIVE,
    enable_offloading=True,
    enable_observation_masking=True,
    cost_per_million_input=0.40 if "mini" in MODEL else 2.00,
    preserve_recent_turns=20,
)

DOMAIN_LIST = ", ".join(ALLOWED_DOMAIN_SUFFIXES)

# ── Report structure lives in the NARRATIVE (system prompt) so it
#    survives compaction.  TASK_TEXT is kept short: data-gathering only.
REPORT_STRUCTURE = (
    "OUTPUT REQUIREMENTS — you MUST follow these EXACTLY:\n\n"
    "Write a PROFESSIONAL equity research report of AT LEAST 4000 words (roughly 5 pages).\n"
    "Reports under 3000 words will be REJECTED by the editor. Do NOT summarize. "
    "Write in full paragraphs with analysis, not bullet points.\n\n"
    "MANDATORY SECTIONS (write ALL 10, with minimum word counts):\n"
    "  1. Executive Summary — 400+ words. FY25 vs FY24 key deltas: revenue, PAT, margin changes.\n"
    "  2. Financial Performance Deep Dive — 600+ words. Revenue, PAT, Operating Margin, Net Margin, "
    "EPS, Cash Flow from Operations, ROCE, ROE. Exact FY24 vs FY25 numbers side-by-side in a table.\n"
    "  3. Segment Analysis — 400+ words. BFSI, Retail, Manufacturing, Life Sciences, Technology & Services. "
    "Revenue contribution %, YoY growth rate for each segment.\n"
    "  4. Geographic Revenue Split — 400+ words. Americas, Europe, UK, India, APAC. "
    "FY24 vs FY25 % contribution and growth. Fastest/slowest growing regions.\n"
    "  5. Digital & AI Strategy — 400+ words. TCS AI.Cloud, GenAI initiatives, large deal wins, "
    "digital revenue as % of total.\n"
    "  6. Human Capital — 300+ words. Employee count, attrition rate, utilization, freshers hired, "
    "training hours, diversity metrics.\n"
    "  7. Competitive Positioning — 400+ words. vs Infosys, Wipro, HCLTech. Revenue comparison, "
    "margin comparison, deal pipeline, market cap.\n"
    "  8. Risk Assessment — 400+ words. Macro risks, technology disruption, currency impact, "
    "geopolitical factors, client concentration, regulatory.\n"
    "  9. Valuation Analysis — 400+ words. P/E ratio, P/B ratio, Dividend Yield, EV/EBITDA, "
    "compare with sector average.\n"
    "  10. Investment Recommendation — 300+ words. 12-month Target Price and rationale.\n\n"
    "RULES:\n"
    "- Every section MUST cite specific numbers with source (e.g. 'Revenue: INR 2,40,893 Cr (AR FY25 p.42)').\n"
    "- Include YoY comparison tables where possible.\n"
    "- Do NOT stop writing until you have covered ALL 10 sections in full.\n"
    "- This is a client-facing report — be thorough, analytical, and professional.\n"
)

TASK_TEXT = (
    "Produce a COMPREHENSIVE equity research report on TCS comparing FY2024 vs FY2025.\n\n"
    "PHASE 1 — Discover PDFs:\n"
    "  Call list_tcs_pdf_inventory.\n\n"
    "PHASE 2 — Deep PDF Analysis (MANDATORY — do ALL 10 reads, do NOT skip any):\n"
    "  Read 1: tcs_annual_report-2024.pdf pages 1-15 (Executive highlights, Chairman's letter)\n"
    "  Read 2: tcs_annual_report-2025.pdf pages 1-15 (Executive highlights, Chairman's letter)\n"
    "  Read 3: tcs_annual_report-2024.pdf pages 15-30 (P&L, balance sheet, cash flow)\n"
    "  Read 4: tcs_annual_report-2025.pdf pages 15-30 (P&L, balance sheet, cash flow)\n"
    "  Read 5: tcs_annual_report-2024.pdf pages 30-50 (Management discussion & analysis)\n"
    "  Read 6: tcs_annual_report-2025.pdf pages 30-50 (Management discussion & analysis)\n"
    "  Read 7: tcs_annual_report-2024.pdf pages 50-70 (Segment-wise, geographic revenue)\n"
    "  Read 8: tcs_annual_report-2025.pdf pages 50-70 (Segment-wise, geographic revenue)\n"
    "  Read 9: tcs_annual_report-2024.pdf pages 100-115 (Notes, key ratios)\n"
    "  Read 10: tcs_annual_report-2025.pdf pages 100-115 (Notes, key ratios)\n\n"
    "PHASE 3 — Live Market Data (fetch BOTH):\n"
    f"  Allowed domains: {DOMAIN_LIST}\n"
    "  Fetch 1: https://www.screener.in/company/TCS/consolidated/\n"
    "  Fetch 2: https://economictimes.indiatimes.com/tata-consultancy-services-ltd/stocks/companyid-8345.cms\n\n"
    "PHASE 4 — AFTER completing ALL 12 tool calls above, write the FULL 4000-5000 word report "
    "following the 10-section structure in your system instructions. "
    "Do NOT start writing until ALL reads and fetches are done.\n"
)

print("=" * 80)
print("  CONFIGURATION")
print("=" * 80)
print(f"\n  Context Management:")
print(f"    Optimal Budget:    {CTX_CONFIG.optimal_budget:,} tokens (quality ceiling)")
print(f"    LLM Context:       {CTX_WINDOW:,} tokens")
print(f"    Budget/Context:    {CTX_CONFIG.optimal_budget/CTX_WINDOW*100:.1f}% of window")
print(f"    ObservationMasker: {CTX_CONFIG.enable_observation_masking}")
print(f"    Offloading:        {CTX_CONFIG.enable_offloading}")
print(f"    Tool Threshold:    {CTX_CONFIG.tool_result_threshold} tokens")
print(f"    Preserve Recent:   {CTX_CONFIG.preserve_recent_turns} turns")
print(f"    Response Reserve:  {CTX_CONFIG.response_reserve:,} tokens (~25K words output)")
print(f"    Cost Rate:         ${CTX_CONFIG.cost_per_million_input}/M input tokens")
print(f"\n  Task:")
print(f"    Length:            {len(TASK_TEXT)} chars")
print(f"    Required reads:    10 PDF reads + 2 web fetches = 12 tool calls minimum")
print(f"    Expected tokens:   ~80K-120K raw, managed down to ~45K budget")

  CONFIGURATION

  Context Management:
    Optimal Budget:    200,000 tokens (quality ceiling)
    LLM Context:       1,047,576 tokens
    Budget/Context:    19.1% of window
    ObservationMasker: True
    Offloading:        True
    Tool Threshold:    500 tokens
    Preserve Recent:   20 turns
    Response Reserve:  32,768 tokens (~25K words output)
    Cost Rate:         $0.4/M input tokens

  Task:
    Length:            1559 chars
    Required reads:    10 PDF reads + 2 web fetches = 12 tool calls minimum
    Expected tokens:   ~80K-120K raw, managed down to ~45K budget


---

## 4. Demo A: Standard Mode — Heavy Load + Context Management + Memory

Standard mode executes a linear tool loop: LLM decides tools, tools execute, results fed back, repeat.

**This demo pushes ~80K-120K tokens of raw data through a 20K budget** — the context engine must
aggressively mask, offload, and compact to keep the LLM's context clean and focused.

- **ContextConfig** with 20K optimal budget (aggressive for 128K context window)
- **SlidingWindowMemory** (window_size=20) for multi-turn follow-ups later
- **Plugins**: ModelCallLimit(60), ToolCallLimit(60), ToolRetry(2)
- **Expected**: 10+ PDF reads, 2+ web fetches, 8+ LLM rounds, ~3-5 min runtime

In [7]:

memory_std = MemoryFactory.create_memory(MemoryStrategy.SLIDING_WINDOW, window_size=20)



agent_std = Agent(
    name="TCS_Standard",
    role="Senior Equity Research Analyst — Indian IT Sector",
    objective="Deliver detailed, data-driven equity research on TCS using real annual reports and live web data.",
    prompt=ZeroShotPrompt().configure(
        system=(
            "You are a senior analyst at a top-tier investment bank writing a client-facing research note.\n\n"
            "DATA GATHERING: You MUST use ALL available tools to gather comprehensive data before writing. "
            "Read EVERY page range listed in the task from EACH PDF — do NOT skip any reads. "
            "Also fetch BOTH web pages listed. Complete ALL 12 tool calls before writing.\n\n"
            + REPORT_STRUCTURE
        ),
        user="Execute the research task below. Gather all data first, then write the full report.",
    ),
    llm=llm,
    tools=ALL_TOOLS,
    memory=memory_std,
    config=AgentConfig(
        execution_mode=ExecutionMode.STANDARD,
        context=CTX_CONFIG,
        enable_tracing=True,
        verbose=True,
        max_tool_calls=60,
        llm_max_output_tokens=32768,
    ),
    plugins=[
        ModelCallLimitPlugin(max_calls=60),
        ToolCallLimitPlugin(max_calls=60),
        ToolRetryPlugin(max_retries=2, base_delay=1.0),
    ],
)

await agent_std.initialize()

print(f"\n  Task:")
print(f"    Length:            {len(TASK_TEXT)} chars")
print(f"    Required reads:    10 PDF reads + 2 web fetches = 12 tool calls minimum")
print(f"    Expected tokens:   ~80K-120K raw, managed down to ~45K budget")


print(f"  Limits:   ModelCalls=60, ToolCalls=60, ToolRetry=2")
print(f"  Tools:    {[getattr(t, 'name', type(t).__name__) for t in ALL_TOOLS]}")

[INFO] TCS_Standard: Initializing agent: TCS_Standard
[DEBUG] TCS_Standard: Plugin manager initialized with 3 plugins
[DEBUG] TCS_Standard: Executor component initialized


[DEBUG] TCS_Standard: Memory system initialized
[DEBUG] TCS_Standard: Prompt system initialized 
 You are a senior analyst at a top-tier investment bank writing a client-facing research note.

DATA GATHERING: You MUST use ALL available tools to gather comprehensive data before writing. Read EVERY page range listed in the task from EACH PDF — do NOT skip any reads. Also fetch BOTH web pages listed. Complete ALL 12 tool calls before writing.

OUTPUT REQUIREMENTS — you MUST follow these EXACTLY:

Write a PROFESSIONAL equity research report of AT LEAST 4000 words (roughly 5 pages).
Reports under 3000 words will be REJECTED by the editor. Do NOT summarize. Write in full paragraphs with analysis, not bullet points.

MANDATORY SECTIONS (write ALL 10, with minimum word counts):
  1. Executive Summary — 400+ words. FY25 vs FY24 key deltas: revenue, PAT, margin changes.
  2. Financial Performance Deep Dive — 600+ words. Revenue, PAT, Operating Margin, Net Margin, EPS, Cash Flow from Operations, 


  Task:
    Length:            1559 chars
    Required reads:    10 PDF reads + 2 web fetches = 12 tool calls minimum
    Expected tokens:   ~80K-120K raw, managed down to ~45K budget
  Limits:   ModelCalls=60, ToolCalls=60, ToolRetry=2
  Tools:    ['list_tcs_pdf_inventory', 'read_annual_report_excerpt', 'fetch_allowlisted_finance_page']


In [8]:
task_std = Task.from_dict({"id": "tcs-standard-001", "objective": TASK_TEXT})

print(f"Task: {task_std.id}")
print(f"Executing Standard mode — watch tool calls reading real PDFs...\n")
print("=" * 100)

result_std = await agent_std.execute(task_std)

print("=" * 100)
print(f"\nStandard Mode complete!")
print(f"  Status:    {result_std.status}")
print(f"  LLM Calls: {len(result_std.llm_calls)}")
print(f"  Duration:  {result_std.duration_ms/1000:.1f}s")

[DEBUG] TCS_Standard: Starting execution for task tcs-standard-001
[INFO] TCS_Standard: Agent 'TCS_Standard' executing in STANDARD mode
[DEBUG] TCS_Standard: Executing in STANDARD mode (tool-enabled, linear)


Task: tcs-standard-001
Executing Standard mode — watch tool calls reading real PDFs...



httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] TCS_Standard: Tool requested: list_tcs_pdf_inventory
[INFO] TCS_Standard: Tool requested: read_annual_report_excerpt



  >>> [TOOL] list_tcs_pdf_inventory()
  >>> [RESULT] 92 chars — 2 PDFs found

  >>> [TOOL] read_annual_report_excerpt('tcs_annual_report-2024.pdf', start=1, pages=15)


nucleusiq.agents.context.engine | DEBUG | Registered read_annual_report_excerpt result (3205 tokens) in ContentStore → read_annual_report_excerpt:cebeb9fef18a (full content kept in context for model)
[INFO] TCS_Standard: Tool requested: read_annual_report_excerpt


...

  >>> [RESULT] 12,015 chars from https://www.screener.in/company/TCS/consolidated/

  >>> [TOOL] fetch_allowlisted_finance_page('https://economictimes.indiatimes.com/tata-consultancy-services-ltd/stocks/compan')


nucleusiq.agents.context.engine | DEBUG | Registered fetch_allowlisted_finance_page result (5003 tokens) in ContentStore → fetch_allowlisted_finance_page:c17a27ebc6e8 (full content kept in context for model)


  >>> [RESULT] 12,015 chars from https://economictimes.indiatimes.com/tata-consultancy-services-ltd/stocks/companyid-8345.cms


httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Standard Mode complete!
  Status:    ResultStatus.SUCCESS
  LLM Calls: 2
  Duration:  86.5s


In [9]:
ct_std = result_std.context_telemetry

def print_context_dashboard(ct, result, title="Context Management Dashboard"):
    """Print a clear, visual context management dashboard."""
    W = 100
    print("=" * W)
    print(f"  {title}")
    print("=" * W)

    if ct is None:
        print("  [No telemetry available]")
        return

    budget = ct.optimal_budget or 20_000
    llm_window = 128_000

    # --- Section 1: Budget vs Reality ---
    print(f"\n  {'─' * 40}")
    print(f"  1. TOKEN BUDGET vs ACTUAL USAGE")
    print(f"  {'─' * 40}")
    total_in = sum(lc.prompt_tokens for lc in result.llm_calls)
    total_out = sum(lc.completion_tokens for lc in result.llm_calls)
    total_tok = total_in + total_out

    budget_bar_len = 50
    used_pct = min(ct.peak_utilization, 1.0)
    used_blocks = int(used_pct * budget_bar_len)
    print(f"  Optimal Budget:    {budget:>8,} tokens")
    print(f"  LLM Window:        {llm_window:>8,} tokens")
    print(f"  Total Tokens Used: {total_tok:>8,} tokens (across {len(result.llm_calls)} LLM calls)")
    print(f"  Peak Utilization:  {ct.peak_utilization:>8.1%}")
    print(f"  Final Utilization: {ct.final_utilization:>8.1%}")
    print()
    print(f"  Budget Usage: [{'█' * used_blocks}{'░' * (budget_bar_len - used_blocks)}] {ct.peak_utilization:.0%}")
    print(f"                 0%{' ' * (budget_bar_len - 9)}100%")

    # --- Section 2: What Context Management DID ---
    print(f"\n  {'─' * 40}")
    print(f"  2. WHAT CONTEXT MANAGEMENT DID")
    print(f"  {'─' * 40}")

    events = []
    if ct.observations_masked > 0:
        events.append(f"  ✓ Observation Masking:  Masked {ct.observations_masked} old tool results, freed {ct.tokens_masked:,} tokens")
    else:
        events.append(f"  · Observation Masking:  No masking needed (few turns)")
    if ct.artifacts_offloaded > 0:
        events.append(f"  ✓ Tool Result Offload:  Offloaded {ct.artifacts_offloaded} large tool results to ContentStore")
    else:
        events.append(f"  · Tool Result Offload:  No offloading triggered")
    if ct.compaction_count > 0:
        events.append(f"  ✓ Compaction Pipeline:  {ct.compaction_count} compaction(s), freed {ct.tokens_freed_total:,} tokens total")
        for ev in ct.compaction_events:
            freed_pct = ev.tokens_freed / ev.tokens_before * 100 if ev.tokens_before > 0 else 0
            events.append(
                f"      [{ev.strategy}] {ev.tokens_before:,} → {ev.tokens_after:,} "
                f"(freed {ev.tokens_freed:,} = {freed_pct:.0f}%, took {ev.duration_ms:.0f}ms)"
            )
    else:
        events.append(f"  · Compaction Pipeline:  No compaction triggered")
    for e in events:
        print(e)

    # --- Section 3: Cost Impact ---
    print(f"\n  {'─' * 40}")
    print(f"  3. COST IMPACT")
    print(f"  {'─' * 40}")
    if ct.estimated_cost_without_mgmt > 0:
        saved = ct.estimated_cost_without_mgmt - ct.estimated_cost_with_mgmt
        print(f"  Without Context Mgmt: ${ct.estimated_cost_without_mgmt:.4f}")
        print(f"  With Context Mgmt:    ${ct.estimated_cost_with_mgmt:.4f}")
        print(f"  Saved:                ${saved:.4f} ({ct.estimated_savings_pct:.1f}%)")
    else:
        print(f"  (Cost estimation not available)")

    # --- Section 4: Turn-by-Turn Token Growth ---
    print(f"\n  {'─' * 40}")
    print(f"  4. TURN-BY-TURN EXECUTION TRACE")
    print(f"  {'─' * 40}")
    cumulative = 0
    max_prompt = max((lc.prompt_tokens for lc in result.llm_calls), default=1)
    for i, lc in enumerate(result.llm_calls):
        cumulative += lc.total_tokens
        bar_len = int(lc.prompt_tokens / max(max_prompt, 1) * 40)
        purpose = f" [{lc.purpose}]" if lc.purpose else ""
        tc_info = f" → {lc.tool_call_count} tools" if lc.has_tool_calls else ""
        print(
            f"  Turn {i+1:>2}: {lc.prompt_tokens:>7,} in + {lc.completion_tokens:>5,} out "
            f"| {'█' * bar_len}{'░' * (40 - bar_len)} "
            f"| {lc.duration_ms/1000:.1f}s{purpose}{tc_info}"
        )
    print(f"  {'─' * 80}")
    print(f"  Total:  {total_in:>7,} in + {total_out:>5,} out = {total_tok:,} tokens | {result.duration_ms/1000:.1f}s")

    # --- Section 5: Context Region Breakdown ---
    if ct.region_breakdown:
        print(f"\n  {'─' * 40}")
        print(f"  5. CONTEXT REGION BREAKDOWN (final snapshot)")
        print(f"  {'─' * 40}")
        total_r = sum(ct.region_breakdown.values()) or 1
        for region, tokens in sorted(ct.region_breakdown.items(), key=lambda x: -x[1]):
            if tokens > 0:
                pct = tokens / total_r * 100
                bar = "█" * int(pct / 2.5)
                print(f"  {region:15s}: {tokens:>7,} tokens ({pct:5.1f}%) {bar}")

    print(f"\n{'=' * W}")

print_context_dashboard(ct_std, result_std, "STANDARD MODE — Context Management Dashboard")

  STANDARD MODE — Context Management Dashboard

  ────────────────────────────────────────
  1. TOKEN BUDGET vs ACTUAL USAGE
  ────────────────────────────────────────
  Optimal Budget:     200,000 tokens
  LLM Window:         128,000 tokens
  Total Tokens Used:   41,465 tokens (across 2 LLM calls)
  Peak Utilization:     23.0%
  Final Utilization:    23.0%

  Budget Usage: [███████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 23%
                 0%                                         100%

  ────────────────────────────────────────
  2. WHAT CONTEXT MANAGEMENT DID
  ────────────────────────────────────────
  · Observation Masking:  No masking needed (few turns)
  ✓ Tool Result Offload:  Offloaded 12 large tool results to ContentStore
  · Compaction Pipeline:  No compaction triggered

  ────────────────────────────────────────
  3. COST IMPACT
  ────────────────────────────────────────
  Without Context Mgmt: $0.0158
  With Context Mgmt:    $0.0158
  Saved:                $0.0000 (

In [10]:
print("=" * 100)
print("  STANDARD MODE — Tool Call Detail")
print("=" * 100)
print()
for i, tc in enumerate(result_std.tool_calls):
    status = "OK  " if tc.success else "FAIL"
    print(f"  [{status}] {i+1:>2}. {tc.tool_name:<40s} {tc.duration_ms:>7,.0f}ms")
print(f"\n  Total tool calls: {len(result_std.tool_calls)}")
print(f"  Failed:           {len(result_std.failed_tool_calls)}")

  STANDARD MODE — Tool Call Detail

  [OK  ]  1. list_tcs_pdf_inventory                         1ms
  [OK  ]  2. read_annual_report_excerpt                   900ms
  [OK  ]  3. read_annual_report_excerpt                 3,731ms
  [OK  ]  4. read_annual_report_excerpt                 2,584ms
  [OK  ]  5. read_annual_report_excerpt                10,786ms
  [OK  ]  6. read_annual_report_excerpt                 2,175ms
  [OK  ]  7. read_annual_report_excerpt                 2,662ms
  [OK  ]  8. read_annual_report_excerpt                 1,014ms
  [OK  ]  9. read_annual_report_excerpt                 3,307ms
  [OK  ] 10. read_annual_report_excerpt                   834ms
  [OK  ] 11. read_annual_report_excerpt                 1,717ms
  [OK  ] 12. fetch_allowlisted_finance_page               649ms
  [OK  ] 13. fetch_allowlisted_finance_page               658ms

  Total tool calls: 13
  Failed:           0


In [11]:
print("=" * 100)
print("  STANDARD MODE — Research Report")
print("=" * 100)
print()
report_std = str(result_std.output)
print(report_std)
print()
print("_" * 100)
words_std = len(report_std.split())
print(f"Report: {len(report_std):,} chars | ~{words_std:,} words | ~{words_std // 250:.0f} pages")

  STANDARD MODE — Research Report

Executive Summary

Tata Consultancy Services (TCS), the flagship IT services company of the Tata Group, reported solid financial and operational performance in fiscal year 2025 (FY25), continuing its trajectory of growth and transformation from FY24. The company’s consolidated revenue increased to INR 2,67,021 crore in FY25, reflecting a 7.09% year-on-year (YoY) growth over INR 2,49,315 crore in FY24 (TCS Annual Reports FY24 pp. 16-18; FY25 pp. 6-8; Screener.in). Profit after tax (PAT) rose to INR 48,553 crore, representing a 5.76% increase from INR 45,908 crore in FY24. Operating margins remained healthy, with a slight moderation from 26.45% in FY24 to 25.89% in FY25, while net profit margins were stable at around 19%. Earnings per share (EPS) advanced to INR 134.19 in FY25 from INR 126.88 in FY24, reflecting consistent profitability and shareholder returns.

The growth in FY25 was driven by broad-based demand across industry verticals and geographie

In [12]:
print(result_std.display())

AgentResult(status=success)
  Agent  : TCS_Standard (85cb6141-bfcd-4246-8c3f-0a27e301f499)
  Task   : tcs-standard-001
  Mode   : standard
  Model  : gpt-4.1-mini
  Time   : 86496.3ms
  Output : Executive Summary

Tata Consultancy Services (TCS), the flagship IT services company of the Tata Group, reported solid financial and operational performance in fiscal year 2025 (FY25), continuing its 
  Tools  : 13 calls
    [OK] list_tcs_pdf_inventory(1ms)
    [OK] read_annual_report_excerpt(900ms)
    [OK] read_annual_report_excerpt(3731ms)
    [OK] read_annual_report_excerpt(2584ms)
    [OK] read_annual_report_excerpt(10786ms)
    [OK] read_annual_report_excerpt(2175ms)
    [OK] read_annual_report_excerpt(2662ms)
    [OK] read_annual_report_excerpt(1014ms)
    [OK] read_annual_report_excerpt(3307ms)
    [OK] read_annual_report_excerpt(834ms)
    [OK] read_annual_report_excerpt(1717ms)
    [OK] fetch_allowlisted_finance_page(649ms)
    [OK] fetch_allowlisted_finance_page(658ms)
  LLM    : 2 c

---

## 4B. PROOF OF VALUE — Baseline (No Context Management) vs Managed

To measure the **real, tangible benefit** of context management, we run the **exact same task** with context management **completely disabled**, then compare.

| Run | Context Engine | What Happens to Tool Results |
|---|---|---|
| **Baseline** | OFF | Full raw text stays in every LLM call. No offloading, masking, or compaction. |
| **Managed** | ON | Large results offloaded (preview kept). Old results masked. Compaction fires if needed. |

**This is the only honest way to prove value — run the same workload both ways and compare.**

In [13]:
print("=" * 100)
print("  BASELINE RUN — NO Context Management (same model, tools, task)")
print("=" * 100)
print("  Context engine: OFF — raw tool results accumulate unmanaged")
print()

agent_baseline = Agent(
    name="TCS_Baseline",
    role="Senior Equity Research Analyst — Indian IT Sector",
    objective="Deliver detailed, data-driven equity research on TCS using real annual reports and live web data.",
    prompt=ZeroShotPrompt().configure(
        system=(
            "You are a senior analyst at a top-tier investment bank writing a client-facing research note.\n\n"
            "DATA GATHERING: You MUST use ALL available tools to gather comprehensive data before writing. "
            "Read EVERY page range listed in the task from EACH PDF — do NOT skip any reads. "
            "Also fetch BOTH web pages listed. Complete ALL 12 tool calls before writing.\n\n"
            + REPORT_STRUCTURE
        ),
        user="Execute the research task below. Gather all data first, then write the full report.",
    ),
    llm=llm,
    tools=ALL_TOOLS,
    config=AgentConfig(
        execution_mode=ExecutionMode.STANDARD,
        respect_context_window=False,
        enable_tracing=True,
        verbose=True,
        max_tool_calls=60,
        llm_max_output_tokens=32768,
    ),
    plugins=[
        ModelCallLimitPlugin(max_calls=60),
        ToolCallLimitPlugin(max_calls=60),
        ToolRetryPlugin(max_retries=2, base_delay=1.0),
    ],
)

await agent_baseline.initialize()

task_bl = Task.from_dict({"id": "tcs-baseline-001", "objective": TASK_TEXT})
print("Executing BASELINE (no context management)...\n")
print("=" * 100)
result_bl = await agent_baseline.execute(task_bl)
print("=" * 100)

print(f"\nBaseline complete!")
print(f"  Status:     {result_bl.status}")
print(f"  LLM Calls:  {len(result_bl.llm_calls)}")
print(f"  Tool Calls: {len(result_bl.tool_calls)}")
print(f"  Duration:   {result_bl.duration_ms/1000:.1f}s")
print(f"  Report:     ~{len(str(result_bl.output).split())} words")

[INFO] TCS_Baseline: Initializing agent: TCS_Baseline
[DEBUG] TCS_Baseline: Plugin manager initialized with 3 plugins
[DEBUG] TCS_Baseline: Executor component initialized
[DEBUG] TCS_Baseline: Prompt system initialized 
 You are a senior analyst at a top-tier investment bank writing a client-facing research note.

DATA GATHERING: You MUST use ALL available tools to gather comprehensive data before writing. Read EVERY page range listed in the task from EACH PDF — do NOT skip any reads. Also fetch BOTH web pages listed. Complete ALL 12 tool calls before writing.

OUTPUT REQUIREMENTS — you MUST follow these EXACTLY:

Write a PROFESSIONAL equity research report of AT LEAST 4000 words (roughly 5 pages).
Reports under 3000 words will be REJECTED by the editor. Do NOT summarize. Write in full paragraphs with analysis, not bullet points.

MANDATORY SECTIONS (write ALL 10, with minimum word counts):
  1. Executive Summary — 400+ words. FY25 vs FY24 key deltas: revenue, PAT, margin changes.
  2.

  BASELINE RUN — NO Context Management (same model, tools, task)
  Context engine: OFF — raw tool results accumulate unmanaged

Executing BASELINE (no context management)...



httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] TCS_Baseline: Tool requested: list_tcs_pdf_inventory



  >>> [TOOL] list_tcs_pdf_inventory()
  >>> [RESULT] 92 chars — 2 PDFs found


httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] TCS_Baseline: Tool requested: read_annual_report_excerpt


...


  >>> [TOOL] fetch_allowlisted_finance_page('https://www.screener.in/company/TCS/consolidated/')


[INFO] TCS_Baseline: Tool requested: fetch_allowlisted_finance_page


  >>> [RESULT] 12,015 chars from https://www.screener.in/company/TCS/consolidated/

  >>> [TOOL] fetch_allowlisted_finance_page('https://economictimes.indiatimes.com/tata-consultancy-services-ltd/stocks/compan')
  >>> [RESULT] 12,015 chars from https://economictimes.indiatimes.com/tata-consultancy-services-ltd/stocks/companyid-8345.cms


httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] TCS_Baseline: Synthesis pass: 19 tool calls over 4 rounds — re-calling LLM without tools for final output
httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Baseline complete!
  Status:     ResultStatus.SUCCESS
  LLM Calls:  6
  Tool Calls: 19
  Duration:   178.8s
  Report:     ~3583 words


In [14]:
W = 100
print("=" * W)
print("  A/B COMPARISON: Baseline (No Context Mgmt) vs Managed (Full Context Mgmt)")
print("=" * W)

bl_calls = len(result_bl.llm_calls)
mg_calls = len(result_std.llm_calls)
bl_in = sum(c.prompt_tokens for c in result_bl.llm_calls)
mg_in = sum(c.prompt_tokens for c in result_std.llm_calls)
bl_out = sum(c.completion_tokens for c in result_bl.llm_calls)
mg_out = sum(c.completion_tokens for c in result_std.llm_calls)
bl_tools = len(result_bl.tool_calls)
mg_tools = len(result_std.tool_calls)
bl_time = result_bl.duration_ms / 1000
mg_time = result_std.duration_ms / 1000
bl_words = len(str(result_bl.output).split())
mg_words = len(str(result_std.output).split())

rate = CTX_CONFIG.cost_per_million_input / 1_000_000
bl_cost = bl_in * rate
mg_cost = mg_in * rate

token_diff = bl_in - mg_in
cost_diff = bl_cost - mg_cost
pct_saved = (token_diff / bl_in * 100) if bl_in > 0 else 0

print(f"\n  {'─' * 40}")
print(f"  1. EXECUTION METRICS (side by side)")
print(f"  {'─' * 40}")
print(f"  {'Metric':<30s} {'Baseline':>15s} {'Managed':>15s} {'Delta':>15s}")
print(f"  {'═' * 75}")
print(f"  {'LLM Calls':<30s} {bl_calls:>15,} {mg_calls:>15,} {mg_calls-bl_calls:>+15,}")
print(f"  {'Tool Calls':<30s} {bl_tools:>15,} {mg_tools:>15,} {mg_tools-bl_tools:>+15,}")
print(f"  {'Total Input Tokens':<30s} {bl_in:>15,} {mg_in:>15,} {mg_in-bl_in:>+15,}")
print(f"  {'Total Output Tokens':<30s} {bl_out:>15,} {mg_out:>15,} {mg_out-bl_out:>+15,}")
print(f"  {'Execution Time':<30s} {bl_time:>14.1f}s {mg_time:>14.1f}s {mg_time-bl_time:>+14.1f}s")
print(f"  {'Report Words':<30s} {bl_words:>15,} {mg_words:>15,} {mg_words-bl_words:>+15,}")
print(f"  {'Cost (input tokens)':<30s} {'$'+f'{bl_cost:.4f}':>15s} {'$'+f'{mg_cost:.4f}':>15s} {'$'+f'{cost_diff:.4f}':>15s}")

# Turn-by-turn comparison
print(f"\n  {'─' * 40}")
print(f"  2. TURN-BY-TURN TOKEN GROWTH")
print(f"  {'─' * 40}")
print(f"  {'Turn':<8s} {'Baseline':>12s} {'Managed':>12s} {'Saved':>12s} {'Visual (baseline=─ managed=█)'}")
max_turns = max(bl_calls, mg_calls)
max_tok = max(
    max((c.prompt_tokens for c in result_bl.llm_calls), default=1),
    max((c.prompt_tokens for c in result_std.llm_calls), default=1),
)
for i in range(max_turns):
    b_tok = result_bl.llm_calls[i].prompt_tokens if i < bl_calls else 0
    m_tok = result_std.llm_calls[i].prompt_tokens if i < mg_calls else 0
    saved = b_tok - m_tok
    b_bar = int(b_tok / max(max_tok, 1) * 30)
    m_bar = int(m_tok / max(max_tok, 1) * 30)
    print(f"  {i+1:<8d} {b_tok:>12,} {m_tok:>12,} {saved:>+12,}  {'─' * b_bar}")
    print(f"  {'':8s} {'':12s} {'':12s} {'':12s}  {'█' * m_bar}")

# Total savings
print(f"\n  {'─' * 40}")
print(f"  3. TOKEN SAVINGS SUMMARY")
print(f"  {'─' * 40}")
print(f"  Baseline total input:   {bl_in:>12,} tokens")
print(f"  Managed total input:    {mg_in:>12,} tokens")
if token_diff > 0:
    print(f"  Tokens SAVED:           {token_diff:>12,} tokens ({pct_saved:.1f}% less)")
    print(f"  Cost SAVED:             ${cost_diff:>11.4f}")
    daily_runs = 100
    print(f"  At {daily_runs} runs/day:          ${cost_diff * daily_runs:>11.2f}/day")
elif token_diff < 0:
    print(f"  Tokens EXTRA:           {abs(token_diff):>12,} tokens ({abs(pct_saved):.1f}% more)")
    print(f"  NOTE: Managed used MORE tokens due to offloading preview overhead.")
    print(f"        The previews keep first 10 lines — for large PDF pages with few")
    print(f"        newlines, the preview can be nearly as large as the original.")
    print(f"        FIX: Reduce preview_lines or add a character cap (v0.7.7 backlog).")
else:
    print(f"  No difference in token usage.")

# What context management did (from managed run telemetry)
print(f"\n  {'─' * 40}")
print(f"  4. WHAT CONTEXT MANAGEMENT DID (Managed run)")
print(f"  {'─' * 40}")
ct = ct_std
if ct:
    print(f"  ✓ Offloaded:   {ct.artifacts_offloaded} large tool results → ContentStore")
    print(f"                 Full content preserved, preview+ref kept in context")
    print(f"  ✓ Masked:      {ct.observations_masked} old results → freed {ct.tokens_masked:,} tokens")
    print(f"  · Compacted:   {ct.compaction_count} events (not triggered — usage was {ct.peak_utilization:.0%} of budget)")
    print(f"  ℹ Peak usage:  {ct.peak_utilization:.1%} of {ct.optimal_budget:,} budget")
    print(f"                 Compaction fires at 70%+ — we stayed well below")

# Verdict
print(f"\n  {'─' * 40}")
print(f"  5. VERDICT — WHEN DOES CONTEXT MANAGEMENT PAY OFF?")
print(f"  {'─' * 40}")
if pct_saved > 10:
    print(f"  ✓ CLEAR WIN: {pct_saved:.0f}% token savings ({token_diff:,} tokens)")
    print(f"    Context management is actively saving cost and reducing latency.")
elif pct_saved > 2:
    print(f"  ~ MODEST WIN: {pct_saved:.1f}% savings ({token_diff:,} tokens)")
    print(f"    Benefit scales with more tool calls and longer conversations.")
else:
    verdict = "NET NEUTRAL" if abs(pct_saved) < 2 else "OVERHEAD (preview too generous)"
    print(f"  ⚠ {verdict} for this specific workload. Here's why:")
    print(f"    Budget: 200K tokens — actual usage: {ct.peak_utilization:.0%} → compaction never needed")
    print(f"    Preserve: last {CTX_CONFIG.preserve_recent_turns} turns — all {mg_tools} results kept")
    print(f"    Preview: first 10 lines per offloaded result — nearly as large as original")
print()
print(f"  CONTEXT MANAGEMENT IS ESSENTIAL WHEN:")
print(f"  ┌──────────────────────────────────────────────────────────────────────────┐")
print(f"  │ Scenario                          │ Why It Matters                       │")
print(f"  ├──────────────────────────────────────────────────────────────────────────┤")
print(f"  │ Small models (GPT-4: 8K)          │ Overflow without compaction          │")
print(f"  │ 50+ tool calls per task            │ Compaction fires, frees real tokens  │")
print(f"  │ Multi-turn (10+ follow-ups)        │ Memory + context grows unbounded     │")
print(f"  │ Production at scale (1M+ req/mo)   │ Even 5% savings = significant $      │")
print(f"  │ Quality-sensitive tasks            │ Masking prevents 'lost in middle'    │")
print(f"  └──────────────────────────────────────────────────────────────────────────┘")

print(f"\n{'=' * W}")

  A/B COMPARISON: Baseline (No Context Mgmt) vs Managed (Full Context Mgmt)

  ────────────────────────────────────────
  1. EXECUTION METRICS (side by side)
  ────────────────────────────────────────
  Metric                                Baseline         Managed           Delta
  ═══════════════════════════════════════════════════════════════════════════
  LLM Calls                                    6               2              -4
  Tool Calls                                  19              13              -6
  Total Input Tokens                     173,648          37,747        -135,901
  Total Output Tokens                     11,145           3,718          -7,427
  Execution Time                          178.8s           86.5s          -92.3s
  Report Words                             3,583           2,051          -1,532
  Cost (input tokens)                    $0.0695         $0.0151         $0.0544

  ────────────────────────────────────────
  2. TURN-BY-TURN TOKEN GROWT

---

## 4C. STRESS TEST — Tight Budget (30K) Forces Compaction

This is the **real proof**. We shrink the budget to **30,000 tokens** — simulating a smaller model like GPT-4 (8K) or GPT-3.5 (16K).

With 19 tool calls generating ~50K+ tokens of data, the context engine **must** compact aggressively to keep the agent running. This is where context management earns its keep.

In [15]:
STRESS_CONFIG = ContextConfig(
    optimal_budget=30_000,
    response_reserve=4_096,
    tool_result_threshold=300,
    strategy=ContextStrategy.PROGRESSIVE,
    enable_offloading=True,
    enable_observation_masking=True,
    cost_per_million_input=CTX_CONFIG.cost_per_million_input,
    preserve_recent_turns=3,
)

print("=" * 100)
print("  STRESS TEST — 30K Budget (simulates small-model constraints)")
print("=" * 100)
print(f"  Budget:           {STRESS_CONFIG.optimal_budget:,} tokens (vs {CTX_CONFIG.optimal_budget:,} in managed)")
print(f"  Preserve recent:  {STRESS_CONFIG.preserve_recent_turns} turns (vs {CTX_CONFIG.preserve_recent_turns} in managed)")
print(f"  Tool threshold:   {STRESS_CONFIG.tool_result_threshold} tokens (vs {CTX_CONFIG.tool_result_threshold} in managed)")
print(f"  Response reserve: {STRESS_CONFIG.response_reserve:,} tokens")
print()

agent_stress = Agent(
    name="TCS_StressTest",
    role="Senior Equity Research Analyst — Indian IT Sector",
    objective="Deliver detailed, data-driven equity research on TCS using real annual reports and live web data.",
    prompt=ZeroShotPrompt().configure(
        system=(
            "You are a senior analyst at a top-tier investment bank writing a client-facing research note.\n\n"
            "DATA GATHERING: You MUST use ALL available tools to gather comprehensive data before writing. "
            "Read EVERY page range listed in the task from EACH PDF — do NOT skip any reads. "
            "Also fetch BOTH web pages listed. Complete ALL 12 tool calls before writing.\n\n"
            + REPORT_STRUCTURE
        ),
        user="Execute the research task below. Gather all data first, then write the full report.",
    ),
    llm=llm,
    tools=ALL_TOOLS,
    config=AgentConfig(
        execution_mode=ExecutionMode.STANDARD,
        context=STRESS_CONFIG,
        enable_tracing=True,
        verbose=True,
        max_tool_calls=60,
        llm_max_output_tokens=4096,
    ),
    plugins=[
        ModelCallLimitPlugin(max_calls=60),
        ToolCallLimitPlugin(max_calls=60),
        ToolRetryPlugin(max_retries=2, base_delay=1.0),
    ],
)

await agent_stress.initialize()

task_stress = Task.from_dict({"id": "tcs-stress-001", "objective": TASK_TEXT})
print("Executing STRESS TEST (30K budget, aggressive compaction)...\n")
print("=" * 100)
result_stress = await agent_stress.execute(task_stress)
print("=" * 100)

print(f"\nStress Test complete!")
print(f"  Status:     {result_stress.status}")
print(f"  LLM Calls:  {len(result_stress.llm_calls)}")
print(f"  Tool Calls: {len(result_stress.tool_calls)}")
print(f"  Duration:   {result_stress.duration_ms/1000:.1f}s")
print(f"  Report:     ~{len(str(result_stress.output).split())} words")

[INFO] TCS_StressTest: Initializing agent: TCS_StressTest
[DEBUG] TCS_StressTest: Plugin manager initialized with 3 plugins
[DEBUG] TCS_StressTest: Executor component initialized
[DEBUG] TCS_StressTest: Prompt system initialized 
 You are a senior analyst at a top-tier investment bank writing a client-facing research note.

DATA GATHERING: You MUST use ALL available tools to gather comprehensive data before writing. Read EVERY page range listed in the task from EACH PDF — do NOT skip any reads. Also fetch BOTH web pages listed. Complete ALL 12 tool calls before writing.

OUTPUT REQUIREMENTS — you MUST follow these EXACTLY:

Write a PROFESSIONAL equity research report of AT LEAST 4000 words (roughly 5 pages).
Reports under 3000 words will be REJECTED by the editor. Do NOT summarize. Write in full paragraphs with analysis, not bullet points.

MANDATORY SECTIONS (write ALL 10, with minimum word counts):
  1. Executive Summary — 400+ words. FY25 vs FY24 key deltas: revenue, PAT, margin cha

  STRESS TEST — 30K Budget (simulates small-model constraints)
  Budget:           30,000 tokens (vs 200,000 in managed)
  Preserve recent:  3 turns (vs 20 in managed)
  Tool threshold:   300 tokens (vs 500 in managed)
  Response reserve: 4,096 tokens

Executing STRESS TEST (30K budget, aggressive compaction)...



httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] TCS_StressTest: Tool requested: list_tcs_pdf_inventory
[INFO] TCS_StressTest: Tool requested: read_annual_report_excerpt



  >>> [TOOL] list_tcs_pdf_inventory()
  >>> [RESULT] 92 chars — 2 PDFs found

  >>> [TOOL] read_annual_report_excerpt('tcs_annual_report-2024.pdf', start=1, pages=15)


nucleusiq.agents.context.engine | DEBUG | Registered read_annual_report_excerpt result (3205 tokens) in ContentStore → read_annual_report_excerpt:575d02da2e62 (full content kept in context for model)
[INFO] TCS_StressTest: Tool requested: read_annual_report_excerpt


...

  >>> [RESULT] 12,015 chars from https://www.screener.in/company/TCS/consolidated/

  >>> [TOOL] fetch_allowlisted_finance_page('https://economictimes.indiatimes.com/tata-consultancy-services-ltd/stocks/compan')


nucleusiq.agents.context.engine | DEBUG | Registered fetch_allowlisted_finance_page result (5003 tokens) in ContentStore → fetch_allowlisted_finance_page:2691f95449f9 (full content kept in context for model)
nucleusiq.agents.context.engine | DEBUG | Context utilization 100.0% of optimal budget — triggering compaction
nucleusiq.agents.context.engine | INFO | Compaction [tool_result_compactor]: freed 32634 tokens (100.0% → 22.4%)


  >>> [RESULT] 12,015 chars from https://economictimes.indiatimes.com/tata-consultancy-services-ltd/stocks/companyid-8345.cms


httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Stress Test complete!
  Status:     ResultStatus.SUCCESS
  LLM Calls:  2
  Tool Calls: 13
  Duration:   81.1s
  Report:     ~2359 words


In [16]:
ct_stress = result_stress.context_telemetry
print_context_dashboard(ct_stress, result_stress, "STRESS TEST — Context Management Dashboard (30K budget)")

# 3-way comparison
print()
print("=" * 100)
print("  FINAL 3-WAY COMPARISON: Baseline vs Managed (200K) vs Stress Test (30K)")
print("=" * 100)

runs = {
    "Baseline\n(no ctx mgmt)": result_bl,
    "Managed\n(200K budget)": result_std,
    "Stress\n(30K budget)": result_stress,
}

print(f"\n  {'─' * 40}")
print(f"  EXECUTION METRICS")
print(f"  {'─' * 40}")
header = f"  {'Metric':<25s}"
for name in runs:
    header += f" {name.split(chr(10))[0]:>15s}"
print(header)
print(f"  {'═' * 70}")

for label, get_fn in [
    ("LLM Calls", lambda r: f"{len(r.llm_calls):,}"),
    ("Tool Calls", lambda r: f"{len(r.tool_calls):,}"),
    ("Input Tokens", lambda r: f"{sum(c.prompt_tokens for c in r.llm_calls):,}"),
    ("Output Tokens", lambda r: f"{sum(c.completion_tokens for c in r.llm_calls):,}"),
    ("Time", lambda r: f"{r.duration_ms/1000:.1f}s"),
    ("Report Words", lambda r: f"{len(str(r.output).split()):,}"),
]:
    row = f"  {label:<25s}"
    for r in runs.values():
        row += f" {get_fn(r):>15s}"
    print(row)

# Cost comparison
print(f"\n  {'─' * 40}")
print(f"  COST COMPARISON (input tokens)")
print(f"  {'─' * 40}")
for name, r in runs.items():
    short = name.split('\n')[0]
    tokens = sum(c.prompt_tokens for c in r.llm_calls)
    cost = tokens * rate
    print(f"  {short:<25s} {tokens:>10,} tokens  ${cost:.4f}")

# Context management activity
print(f"\n  {'─' * 40}")
print(f"  CONTEXT MANAGEMENT ACTIVITY")
print(f"  {'─' * 40}")
print(f"  {'Feature':<25s} {'Baseline':>15s} {'Managed':>15s} {'Stress':>15s}")
print(f"  {'═' * 70}")

ct_bl_data = result_bl.context_telemetry
ct_mg_data = ct_std
ct_st_data = ct_stress

for label, get_fn in [
    ("Offloaded", lambda ct: f"{ct.artifacts_offloaded}" if ct else "—"),
    ("Masked", lambda ct: f"{ct.observations_masked}" if ct else "—"),
    ("Tokens Freed", lambda ct: f"{ct.tokens_freed_total:,}" if ct else "—"),
    ("Compaction Events", lambda ct: f"{ct.compaction_count}" if ct else "—"),
    ("Peak Utilization", lambda ct: f"{ct.peak_utilization:.0%}" if ct else "—"),
    ("Cost Savings %", lambda ct: f"{ct.estimated_savings_pct:.1f}%" if ct else "—"),
]:
    row = f"  {label:<25s}"
    for ct_data in [ct_bl_data, ct_mg_data, ct_st_data]:
        row += f" {get_fn(ct_data):>15s}"
    print(row)

# Compaction details for stress test
if ct_st_data and ct_st_data.compaction_count > 0:
    print(f"\n  STRESS TEST COMPACTION DETAILS:")
    for i, ev in enumerate(ct_st_data.compaction_events):
        freed_pct = ev.tokens_freed / ev.tokens_before * 100 if ev.tokens_before else 0
        print(
            f"    [{i+1}] {ev.strategy}: {ev.tokens_before:,} → {ev.tokens_after:,} "
            f"(freed {ev.tokens_freed:,} = {freed_pct:.0f}%, took {ev.duration_ms:.0f}ms)"
        )

# Verdict
print(f"\n  {'─' * 40}")
print(f"  VERDICT")
print(f"  {'─' * 40}")

stress_compactions = ct_st_data.compaction_count if ct_st_data else 0
stress_freed = ct_st_data.tokens_freed_total if ct_st_data else 0

if stress_compactions > 0:
    print(f"  ✓ STRESS TEST PROVES THE VALUE:")
    print(f"    With a 30K budget, the compaction pipeline fired {stress_compactions} time(s),")
    print(f"    freeing {stress_freed:,} tokens. The agent COMPLETED the task despite having")
    print(f"    a budget far smaller than the raw data volume (~50K+ tokens).")
    print(f"    Without context management, this would FAIL or produce garbage output.")
    print()
    print(f"  The 200K budget run shows context management as a SAFETY NET —")
    print(f"  it doesn't actively save because there's no pressure.")
    print(f"  The 30K budget run shows it as an ACTIVE OPTIMIZER —")
    print(f"  it aggressively manages context to keep the agent functional.")
else:
    print(f"  ⚠ Stress test compaction did not fire. The agent handled the load")
    print(f"    within the 30K budget through offloading alone.")

print(f"\n  KEY TAKEAWAY:")
print(f"  Context management is like insurance — invisible when things are fine,")
print(f"  but critical when limits are hit. The tighter the budget (smaller model,")
print(f"  more tools, longer conversations), the more value it delivers.")

print(f"\n{'=' * 100}")

  STRESS TEST — Context Management Dashboard (30K budget)

  ────────────────────────────────────────
  1. TOKEN BUDGET vs ACTUAL USAGE
  ────────────────────────────────────────
  Optimal Budget:      30,000 tokens
  LLM Window:         128,000 tokens
  Total Tokens Used:   11,576 tokens (across 2 LLM calls)
  Peak Utilization:    100.0%
  Final Utilization:    22.4%

  Budget Usage: [██████████████████████████████████████████████████] 100%
                 0%                                         100%

  ────────────────────────────────────────
  2. WHAT CONTEXT MANAGEMENT DID
  ────────────────────────────────────────
  · Observation Masking:  No masking needed (few turns)
  ✓ Tool Result Offload:  Offloaded 24 large tool results to ContentStore
  ✓ Compaction Pipeline:  1 compaction(s), freed 32,634 tokens total
      [tool_result_compactor] 38,427 → 5,793 (freed 32,634 = 85%, took 108ms)

  ────────────────────────────────────────
  3. COST IMPACT
  ─────────────────────────────

---

## 5. Demo B: Autonomous Mode — WITH Context Management + Memory

Autonomous mode **decomposes** the task into parallel sub-agents, validates with a Critic,
and synthesizes the final report. **Sub-agent metrics (LLM calls, tool calls, context
telemetry) are now rolled up** into the parent result — giving full visibility.

Same tools, same context config, same task — different execution strategy.

- **Sub-agents**: Each runs in Standard mode with isolated context
- **Telemetry rollup**: v0.7.6 aggregates sub-agent LLM calls, tool calls, and context events
- **Plugins**: ModelCallLimit(80), ToolCallLimit(80), ToolRetry(2)

In [17]:
memory_auto = MemoryFactory.create_memory(MemoryStrategy.SLIDING_WINDOW, window_size=20)

agent_auto = Agent(
    name="TCS_Autonomous",
    role="Senior Equity Research Analyst — Indian IT Sector",
    objective="Deliver detailed, data-driven equity research on TCS using real annual reports and live web data.",
    prompt=ZeroShotPrompt().configure(
        system=(
            "You are a senior analyst at a top-tier investment bank writing a client-facing research note.\n\n"
            "DATA GATHERING: You MUST use ALL available tools to gather comprehensive data before writing. "
            "Read EVERY page range listed in the task from EACH PDF — do NOT skip any reads. "
            "Also fetch BOTH web pages listed. Complete ALL 12 tool calls before writing.\n\n"
            + REPORT_STRUCTURE
        ),
        user="Execute the research task below. Gather all data first, then write the full report.",
    ),
    llm=llm,
    tools=ALL_TOOLS,
    memory=memory_auto,
    config=AgentConfig(
        execution_mode=ExecutionMode.AUTONOMOUS,
        context=CTX_CONFIG,
        enable_tracing=True,
        verbose=True,
        max_tool_calls=80,
        llm_max_output_tokens=32768,
    ),
    plugins=[
        ModelCallLimitPlugin(max_calls=80),
        ToolCallLimitPlugin(max_calls=80),
        ToolRetryPlugin(max_retries=2, base_delay=1.0),
    ],
)

await agent_auto.initialize()

print("Autonomous Mode Agent ready:")
print(f"  Model:    {llm.model_name}")
print(f"  Mode:     {agent_auto.config.execution_mode}")
print(f"  Memory:   SlidingWindow (size=20)")
print(f"  Output:   max_output_tokens=32768 (~25K words)")
print(f"  Limits:   ModelCalls=80, ToolCalls=80, ToolRetry=2")
print(f"  NOTE:     Sub-agent metrics will be rolled up into parent telemetry")

[INFO] TCS_Autonomous: Initializing agent: TCS_Autonomous
[DEBUG] TCS_Autonomous: Plugin manager initialized with 3 plugins
[DEBUG] TCS_Autonomous: Executor component initialized
[DEBUG] TCS_Autonomous: Memory system initialized
[DEBUG] TCS_Autonomous: Prompt system initialized 
 You are a senior analyst at a top-tier investment bank writing a client-facing research note.

DATA GATHERING: You MUST use ALL available tools to gather comprehensive data before writing. Read EVERY page range listed in the task from EACH PDF — do NOT skip any reads. Also fetch BOTH web pages listed. Complete ALL 12 tool calls before writing.

OUTPUT REQUIREMENTS — you MUST follow these EXACTLY:

Write a PROFESSIONAL equity research report of AT LEAST 4000 words (roughly 5 pages).
Reports under 3000 words will be REJECTED by the editor. Do NOT summarize. Write in full paragraphs with analysis, not bullet points.

MANDATORY SECTIONS (write ALL 10, with minimum word counts):
  1. Executive Summary — 400+ words.

Autonomous Mode Agent ready:
  Model:    gpt-4.1-mini
  Mode:     ExecutionMode.AUTONOMOUS
  Memory:   SlidingWindow (size=20)
  Output:   max_output_tokens=32768 (~25K words)
  Limits:   ModelCalls=80, ToolCalls=80, ToolRetry=2
  NOTE:     Sub-agent metrics will be rolled up into parent telemetry


In [18]:
task_auto = Task.from_dict({"id": "tcs-autonomous-001", "objective": TASK_TEXT})

print(f"Task: {task_auto.id}")
print(f"Executing Autonomous mode — watch decomposition + sub-agents + tool calls...\n")
print("=" * 100)

result_auto = await agent_auto.execute(task_auto)

print("=" * 100)
print(f"\nAutonomous Mode complete!")
print(f"  Status:      {result_auto.status}")
print(f"  LLM Calls:   {len(result_auto.llm_calls)}  (includes sub-agent calls — rolled up)")
print(f"  Tool Calls:  {len(result_auto.tool_calls)}  (includes sub-agent tools — rolled up)")
print(f"  Duration:    {result_auto.duration_ms/1000:.1f}s")

sub_calls = [lc for lc in result_auto.llm_calls if lc.purpose.startswith("sub-agent")]
parent_calls = [lc for lc in result_auto.llm_calls if not lc.purpose.startswith("sub-agent")]
print(f"\n  Breakdown:")
print(f"    Parent LLM calls:    {len(parent_calls)}")
print(f"    Sub-agent LLM calls: {len(sub_calls)}")
print(f"    Total tokens (all):  {sum(lc.total_tokens for lc in result_auto.llm_calls):,}")

[DEBUG] TCS_Autonomous: Starting execution for task tcs-autonomous-001
[INFO] TCS_Autonomous: Agent 'TCS_Autonomous' executing in AUTONOMOUS mode
[DEBUG] TCS_Autonomous: Executing in AUTONOMOUS mode


Task: tcs-autonomous-001
Executing Autonomous mode — watch decomposition + sub-agents + tool calls...



httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] TCS_Autonomous: Task classified as SIMPLE — standard + validate + Critic
[INFO] TCS_Autonomous: Attempt 1/3 [EXECUTE]
httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] TCS_Autonomous: Tool requested: list_tcs_pdf_inventory
[INFO] TCS_Autonomous: Tool requested: read_annual_report_excerpt



  >>> [TOOL] list_tcs_pdf_inventory()
  >>> [RESULT] 92 chars — 2 PDFs found

  >>> [TOOL] read_annual_report_excerpt('tcs_annual_report-2024.pdf', start=1, pages=15)


nucleusiq.agents.context.engine | DEBUG | Registered read_annual_report_excerpt result (3205 tokens) in ContentStore → read_annual_report_excerpt:84dd42760122 (full content kept in context for model)
[INFO] TCS_Autonomous: Tool requested: read_annual_report_excerpt


...

  >>> [RESULT] 12,015 chars from https://www.screener.in/company/TCS/consolidated/

  >>> [TOOL] fetch_allowlisted_finance_page('https://economictimes.indiatimes.com/tata-consultancy-services-ltd/stocks/compan')


nucleusiq.agents.context.engine | DEBUG | Registered fetch_allowlisted_finance_page result (5003 tokens) in ContentStore → fetch_allowlisted_finance_page:3dd1b5ccce21 (full content kept in context for model)


  >>> [RESULT] 12,015 chars from https://economictimes.indiatimes.com/tata-consultancy-services-ltd/stocks/companyid-8345.cms


httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] TCS_Autonomous: Attempt 1/3 [VALIDATE]: valid=True layer=all
httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] TCS_Autonomous: Attempt 1/3 [CRITIC]: PASS (score=1.00)



Autonomous Mode complete!
  Status:      ResultStatus.SUCCESS
  LLM Calls:   4  (includes sub-agent calls — rolled up)
  Tool Calls:  13  (includes sub-agent tools — rolled up)
  Duration:    91.7s

  Breakdown:
    Parent LLM calls:    4
    Sub-agent LLM calls: 0
    Total tokens (all):  46,504


In [19]:
ct_auto = result_auto.context_telemetry
print_context_dashboard(ct_auto, result_auto, "AUTONOMOUS MODE — Context Management Dashboard (with sub-agent rollup)")

  AUTONOMOUS MODE — Context Management Dashboard (with sub-agent rollup)

  ────────────────────────────────────────
  1. TOKEN BUDGET vs ACTUAL USAGE
  ────────────────────────────────────────
  Optimal Budget:     200,000 tokens
  LLM Window:         128,000 tokens
  Total Tokens Used:   45,745 tokens (across 4 LLM calls)
  Peak Utilization:     23.0%
  Final Utilization:    23.0%

  Budget Usage: [███████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 23%
                 0%                                         100%

  ────────────────────────────────────────
  2. WHAT CONTEXT MANAGEMENT DID
  ────────────────────────────────────────
  · Observation Masking:  No masking needed (few turns)
  ✓ Tool Result Offload:  Offloaded 12 large tool results to ContentStore
  · Compaction Pipeline:  No compaction triggered

  ────────────────────────────────────────
  3. COST IMPACT
  ────────────────────────────────────────
  Without Context Mgmt: $0.0158
  With Context Mgmt:    $0.0158
  Saved

In [20]:
print("=" * 100)
print("  AUTONOMOUS MODE — Research Report")
print("=" * 100)
print()
report_auto = str(result_auto.output)
print(report_auto)
print()
print("_" * 100)
words_auto = len(report_auto.split())
print(f"Report: {len(report_auto):,} chars | ~{words_auto:,} words | ~{words_auto // 250:.0f} pages")
print()
print(result_auto.display())

  AUTONOMOUS MODE — Research Report

EXECUTIVE SUMMARY

Tata Consultancy Services (TCS), a leading global IT services and consulting company, demonstrated solid financial growth and operational resilience in fiscal year 2025 (FY25), continuing its trajectory of strong performance from FY24. The company’s consolidated revenues increased from INR 2,40,893 crore in FY24 to INR 2,67,021 crore in FY25, representing a robust year-on-year (YoY) growth of approximately 10.9% (Source: AR FY24 p.16; AR FY25 p.7; Screener.in). Profit after tax (PAT) also recorded a healthy increase, rising from INR 45,908 crore in FY24 to INR 48,553 crore in FY25, marking a 5.7% growth (Source: AR FY24 p.16; AR FY25 p.7). The operating margin exhibited a slight moderation, moving from 26.45% in FY24 to 25.89% in FY25, reflecting investments in digital capabilities and talent, alongside inflationary pressures (Source: AR FY24 p.17; AR FY25 p.7). Net profit margin remained stable at 19.01% in FY25 compared to 19.05

---

## 6. Follow-Up Questions — Memory vs Tools (Explicit Tracking)

The Standard agent has **SlidingWindowMemory** — it remembers the research conversation.
We ask 3 follow-up questions and **explicitly track** for each:

| What We Track | Why It Matters |
|---|---|
| **Tool calls?** | Did the agent re-read PDFs or answer from memory? |
| **Memory messages** | How many prior turns are in context? |
| **Context budget usage** | Is the engine compacting to fit within 20K budget? |
| **Masking events** | Are old tool results being masked to save tokens? |
| **Compaction events** | Did the conversation trigger Tier 2 compaction? |

In [21]:
FOLLOW_UPS = [
    "What is TCS's attrition rate and how does it compare to the industry average?",
    "Summarize the top 3 risks for a TCS investor in one paragraph each.",
    "What was the dividend per share for FY2024 and FY2025? Is TCS a good dividend stock?",
]

ctx_before = memory_std.get_context()
mem_before = len(ctx_before)
print(f"Memory before follow-ups: {mem_before} messages\n")

followup_results = []
for i, question in enumerate(FOLLOW_UPS, 1):
    print(f"{'='*100}")
    print(f"  FOLLOW-UP {i}: {question}")
    print(f"{'='*100}")

    task_fu = Task.from_dict({"id": f"followup-{i}", "objective": question})
    result_fu = await agent_std.execute(task_fu)
    followup_results.append(result_fu)

    tool_count = len(result_fu.tool_calls)
    llm_count = len(result_fu.llm_calls)
    fu_tokens = sum(c.prompt_tokens for c in result_fu.llm_calls)
    ct_fu = result_fu.context_telemetry
    ctx_now = memory_std.get_context()
    mem_now = len(ctx_now)

    print(f"\n  {'─' * 60}")
    print(f"  DATA SOURCE:")
    if tool_count > 0:
        tool_names = [tc.tool_name for tc in result_fu.tool_calls]
        print(f"    → TOOLS CALLED: {tool_count} calls — {tool_names}")
        print(f"    → Agent did NOT have enough info in memory, fetched fresh data")
    else:
        print(f"    → MEMORY ONLY: No tool calls — answered entirely from conversation history")
        print(f"    → Agent used {mem_before} messages from SlidingWindowMemory as context")

    print(f"\n  EXECUTION:")
    print(f"    LLM Calls:     {llm_count}")
    print(f"    Input Tokens:  {fu_tokens:,}")
    print(f"    Duration:      {result_fu.duration_ms/1000:.1f}s")

    print(f"\n  CONTEXT MANAGEMENT:")
    if ct_fu:
        peak_bar_len = 30
        peak_blocks = int(min(ct_fu.peak_utilization, 1.0) * peak_bar_len)
        print(f"    Budget Usage:  [{'█' * peak_blocks}{'░' * (peak_bar_len - peak_blocks)}] {ct_fu.peak_utilization:.0%}")
        print(f"    Masked:        {ct_fu.observations_masked} tool results ({ct_fu.tokens_masked:,} tokens freed)")
        print(f"    Offloaded:     {ct_fu.artifacts_offloaded} artifacts")
        print(f"    Compactions:   {ct_fu.compaction_count}")
        if ct_fu.compaction_events:
            for ev in ct_fu.compaction_events:
                print(f"      ↳ [{ev.strategy}] {ev.tokens_before:,} → {ev.tokens_after:,} (freed {ev.tokens_freed:,})")

    print(f"\n  MEMORY GROWTH:")
    print(f"    Before: {mem_before} messages → After: {mem_now} messages (+{mem_now - mem_before})")
    mem_before = mem_now
    ctx_before = ctx_now

    print(f"\n  ANSWER (preview):")
    answer = str(result_fu.output)
    print(f"    {answer[:400]}{'...' if len(answer) > 400 else ''}")
    print()

[DEBUG] TCS_Standard: Starting execution for task followup-1
[INFO] TCS_Standard: Agent 'TCS_Standard' executing in STANDARD mode
[DEBUG] TCS_Standard: Executing in STANDARD mode (tool-enabled, linear)


Memory before follow-ups: 2 messages

  FOLLOW-UP 1: What is TCS's attrition rate and how does it compare to the industry average?


httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] TCS_Standard: Tool requested: list_tcs_pdf_inventory
[INFO] TCS_Standard: Tool requested: fetch_allowlisted_finance_page



  >>> [TOOL] list_tcs_pdf_inventory()
  >>> [RESULT] 92 chars — 2 PDFs found

  >>> [TOOL] fetch_allowlisted_finance_page('https://www.screener.in/company/TCS/consolidated/')


nucleusiq.agents.context.engine | DEBUG | Registered fetch_allowlisted_finance_page result (4734 tokens) in ContentStore → fetch_allowlisted_finance_page:cab7895195e7 (full content kept in context for model)
[INFO] TCS_Standard: Tool requested: fetch_allowlisted_finance_page


...


  >>> [TOOL] read_annual_report_excerpt('tcs_annual_report-2024.pdf', start=8, pages=2)


nucleusiq.agents.context.engine | DEBUG | Registered read_annual_report_excerpt result (1876 tokens) in ContentStore → read_annual_report_excerpt:5e6802000be7 (full content kept in context for model)


  >>> [RESULT] 7,649 chars from pages 8-9 of 341 total


httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
nucleusiq.agents.context.engine | INFO | ObservationMasker: masked 1 tool results, freed 761 tokens
[INFO] TCS_Standard: Synthesis pass: 12 tool calls over 10 rounds — re-calling LLM without tools for final output
httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
nucleusiq.agents.context.engine | INFO | ObservationMasker: masked 1 tool results, freed 760 tokens



  ────────────────────────────────────────────────────────────
  DATA SOURCE:
    → TOOLS CALLED: 12 calls — ['list_tcs_pdf_inventory', 'fetch_allowlisted_finance_page', 'fetch_allowlisted_finance_page', 'read_annual_report_excerpt', 'read_annual_report_excerpt', 'read_annual_report_excerpt', 'read_annual_report_excerpt', 'read_annual_report_excerpt', 'read_annual_report_excerpt', 'read_annual_report_excerpt', 'read_annual_report_excerpt', 'read_annual_report_excerpt']
    → Agent did NOT have enough info in memory, fetched fresh data

  EXECUTION:
    LLM Calls:     12
    Input Tokens:  177,258
    Duration:      112.4s

  CONTEXT MANAGEMENT:
    Budget Usage:  [███░░░░░░░░░░░░░░░░░░░░░░░░░░░] 13%
    Masked:        12 tool results (27,590 tokens freed)
    Offloaded:     23 artifacts
    Compactions:   0

  MEMORY GROWTH:
    Before: 6 messages → After: 8 messages (+2)

  ANSWER (preview):
    Tata Consultancy Services (TCS)  
Comprehensive Equity Research Report: FY2024 vs FY2025 

---

## 7. Head-to-Head Comparison

In [22]:
W = 100
print("=" * W)
print("  FINAL COMPARISON: Standard vs Autonomous (with sub-agent rollup)")
print("=" * W)

s_calls = len(result_std.llm_calls)
a_calls = len(result_auto.llm_calls)
s_in = sum(c.prompt_tokens for c in result_std.llm_calls)
a_in = sum(c.prompt_tokens for c in result_auto.llm_calls)
s_out = sum(c.completion_tokens for c in result_std.llm_calls)
a_out = sum(c.completion_tokens for c in result_auto.llm_calls)
s_tools = len(result_std.tool_calls)
a_tools = len(result_auto.tool_calls)
s_time = result_std.duration_ms / 1000
a_time = result_auto.duration_ms / 1000
s_words = len(str(result_std.output).split())
a_words = len(str(result_auto.output).split())

gpt4o_in = 2.50 / 1_000_000
gpt4o_out = 10.0 / 1_000_000
s_cost = s_in * gpt4o_in + s_out * gpt4o_out
a_cost = a_in * gpt4o_in + a_out * gpt4o_out

print(f"\n  {'─' * 40}")
print(f"  EXECUTION COMPARISON")
print(f"  {'─' * 40}")
print(f"  {'Metric':<35s} {'Standard':>15s} {'Autonomous':>15s} {'Winner':>10s}")
print(f"  {'═'*80}")
print(f"  {'LLM Calls':<35s} {s_calls:>15,} {a_calls:>15,} {'◄' if s_calls < a_calls else '►' if s_calls > a_calls else '='  :>10s}")
print(f"  {'Tool Calls':<35s} {s_tools:>15,} {a_tools:>15,} {'◄' if s_tools < a_tools else '►' if s_tools > a_tools else '=':>10s}")
print(f"  {'Total Input Tokens':<35s} {s_in:>15,} {a_in:>15,} {'◄' if s_in < a_in else '►' if s_in > a_in else '=':>10s}")
print(f"  {'Total Output Tokens':<35s} {s_out:>15,} {a_out:>15,}")
print(f"  {'Total Tokens':<35s} {s_in+s_out:>15,} {a_in+a_out:>15,}")
print(f"  {'Execution Time':<35s} {s_time:>14.1f}s {a_time:>14.1f}s {'◄' if s_time < a_time else '►':>10s}")
print(f"  {'Report Words':<35s} {s_words:>15,} {a_words:>15,} {'◄' if s_words > a_words else '►':>10s}")
print(f"  {'Estimated Cost (input)':<35s} {'$'+f'{s_cost:.4f}':>15s} {'$'+f'{a_cost:.4f}':>15s} {'◄' if s_cost < a_cost else '►':>10s}")

sub_calls = [lc for lc in result_auto.llm_calls if lc.purpose.startswith("sub-agent")]
parent_calls = [lc for lc in result_auto.llm_calls if not lc.purpose.startswith("sub-agent")]
if sub_calls:
    print(f"\n  Autonomous Breakdown (sub-agent rollup):")
    print(f"    Parent LLM calls:    {len(parent_calls)} (decomposer + synthesis)")
    print(f"    Sub-agent LLM calls: {len(sub_calls)} (parallel research)")
    print(f"    Sub-agent tokens:    {sum(lc.total_tokens for lc in sub_calls):,}")

print(f"\n  {'─' * 40}")
print(f"  CONTEXT MANAGEMENT COMPARISON")
print(f"  {'─' * 40}")
print(f"  {'Metric':<35s} {'Standard':>15s} {'Autonomous':>15s}")
print(f"  {'═'*70}")
for metric, get_fn in [
    ("Peak Utilization", lambda ct: f"{ct.peak_utilization:.0%}" if ct else "N/A"),
    ("Observations Masked", lambda ct: f"{ct.observations_masked}" if ct else "N/A"),
    ("Tokens Freed (masking)", lambda ct: f"{ct.tokens_masked:,}" if ct else "N/A"),
    ("Artifacts Offloaded", lambda ct: f"{ct.artifacts_offloaded}" if ct else "N/A"),
    ("Compaction Events", lambda ct: f"{ct.compaction_count}" if ct else "N/A"),
    ("Total Tokens Freed", lambda ct: f"{ct.tokens_freed_total:,}" if ct else "N/A"),
    ("Cost Savings %", lambda ct: f"{ct.estimated_savings_pct:.1f}%" if ct else "N/A"),
]:
    print(f"  {metric:<35s} {get_fn(ct_std):>15s} {get_fn(ct_auto):>15s}")

print(f"\n  {'─' * 40}")
print(f"  FOLLOW-UP QUESTIONS (Standard agent)")
print(f"  {'─' * 40}")
for i, r in enumerate(followup_results, 1):
    fu_in = sum(c.prompt_tokens for c in r.llm_calls)
    ct_fu = r.context_telemetry
    tc_count = len(r.tool_calls)
    source = f"TOOLS ({tc_count})" if tc_count > 0 else "MEMORY ONLY"
    masked = ct_fu.observations_masked if ct_fu else 0
    compacted = ct_fu.compaction_count if ct_fu else 0
    print(f"  Q{i}: {source:<15s} | {fu_in:>6,} tokens | {len(r.llm_calls)} LLM | mask={masked} compact={compacted} | {r.duration_ms/1000:.1f}s")

total_fu_tokens = sum(sum(c.prompt_tokens for c in r.llm_calls) for r in followup_results)
final_mem = memory_std.get_context()
print(f"\n  Total follow-up tokens: {total_fu_tokens:,}")
print(f"  Final memory size:      {len(final_mem)} messages")

print(f"\n{'=' * W}")

  FINAL COMPARISON: Standard vs Autonomous (with sub-agent rollup)

  ────────────────────────────────────────
  EXECUTION COMPARISON
  ────────────────────────────────────────
  Metric                                     Standard      Autonomous     Winner
  ════════════════════════════════════════════════════════════════════════════════
  LLM Calls                                         2               4          ◄
  Tool Calls                                       13              13          =
  Total Input Tokens                           37,747          40,677          ◄
  Total Output Tokens                           3,718           5,068
  Total Tokens                                 41,465          45,745
  Execution Time                                86.5s           91.7s          ◄
  Report Words                                  2,051           2,346          ►
  Estimated Cost (input)                      $0.1315         $0.1524          ◄

  ──────────────────────────────

---

## Summary — What This Notebook Proves

### The Three-Run Comparison

| Run | Context Mgmt | Result | Key Insight |
|---|---|---|---|
| **Managed (200K)** | Full | Safety net mode — offloading active, compaction idle | Works as insurance; minimal savings when budget > usage |
| **Baseline (none)** | OFF | Raw accumulation — all tool results kept in full | Shows the raw token growth without management |
| **Stress (30K)** | Aggressive | Compaction fires — tokens actively freed | Proves the system works under real pressure |

### When Context Management Pays Off

| Scenario | Budget Pressure | Benefit |
|---|---|---|
| Large model, few tools (this demo's Managed run) | Low (~32%) | Monitoring + safety net |
| Large model, many tools (50+ calls) | Medium (~70%+) | Compaction fires, real savings |
| Small model (GPT-4 8K, GPT-3.5 16K) | High (>100%) | **Essential** — overflow without it |
| Multi-turn sessions (10+ follow-ups) | Growing | Memory + compaction prevents runaway |
| Production at scale (1M+ requests) | Any | Even 5% savings = significant $ |

### How the Pipeline Works

```
Per LLM call:
  1. Tool executes (reads PDF / fetches web) → engine.ingest_tool_result()
     └─ If result > threshold: offload to ContentStore, keep preview in context
  2. Before LLM call                         → engine.prepare() [Tiers 1-3]
     └─ If utilization > trigger: run compaction (conversation → emergency)
  3. LLM responds
  4. After LLM response                      → engine.post_response() [Tier 0]
     └─ Mask old tool results that the LLM has already processed
  5. Memory stores the exchange
  6. Next iteration with cleaner, focused context
```

### How to Read the Dashboards

- **Budget Usage bar**: % of optimal budget used. Higher = more aggressive management needed.
- **Turn-by-turn trace**: Token growth per LLM call. Compare Baseline vs Managed to see savings.
- **Region breakdown**: Where tokens are spent (tool_result, user, assistant, system, reserved).
- **Compaction events**: Before/after token counts with strategy used and time taken.
- **A/B Delta column**: Positive = baseline used more (savings). Negative = managed used more (overhead).

---

*NucleusIQ v0.7.6 — Context Window Management*

*Disclaimer: TCS financial data from real annual reports. Not investment advice.*